In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 1. Parameters for the simulation
initial_capital = 1000000
annual_withdrawal = 55000  # Annual withdrawal
years = 25

# Fixed return set (identical for both scenarios: 22x gain, 3x loss)
# This ensures the average return is exactly the same.
base_returns = [0.07] * 22 + [-0.20, -0.15, -0.12] 

# Scenario A: the bad years come first (bad sequence)
ret_bad_start = sorted(base_returns)  

# Scenario B: the good years come first (good sequence)
ret_good_start = sorted(base_returns, reverse=True)  

def simulate_portfolio(initial, returns, withdrawal=0):
    balances = [initial]
    current_balance = initial
    for r in returns:
        # The market return applies first, then the withdrawal
        current_balance = (current_balance * (1 + r)) - withdrawal
        # The portfolio cannot fall below 0
        balances.append(max(0, current_balance))
    return balances

# Run the simulations
path_a_no_ext = simulate_portfolio(initial_capital, ret_bad_start, 0)
path_b_no_ext = simulate_portfolio(initial_capital, ret_good_start, 0)
path_a_ext = simulate_portfolio(initial_capital, ret_bad_start, annual_withdrawal)
path_b_ext = simulate_portfolio(initial_capital, ret_good_start, annual_withdrawal)

# 2. Build the figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 10), sharex=True)
plt.style.use('seaborn-v0_8-whitegrid')

# Helper for currency formatting (e.g. 1.000.000 € with dot thousands separator)
def euro_format(x, p):
    return f"{int(x):,}".replace(",", ".") + " €"

# --- TOP PLOT: without withdrawals ---
ax1.plot(path_a_no_ext, color='#d62728', linestyle='--', alpha=0.6, label='Scenario A: early bear market (losses at the start)')
ax1.plot(path_b_no_ext, color='#1f77b4', linestyle='--', alpha=0.6, label='Scenario B: late bear market (losses at the end)')
ax1.set_title('A: without withdrawals (pure compounding / accumulation)', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Portfolio value (€)', fontsize=10)

# Optimized annotation with background box for readability
ax1.annotate(f'Final values identical:\n{path_a_no_ext[-1]:,.0f} €'.replace(',', '.'), 
             xy=(years, path_a_no_ext[-1]), 
             #xytext=(years-11, 3_800_000), # position in whitespace
             xytext=(years-6, 2_800_000), # position in whitespace
             fontsize=10,
             fontweight='bold',
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=7),
             bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='gray', alpha=0.9))

# --- BOTTOM PLOT: with withdrawals (SORR) ---
ax2.plot(path_a_ext, color='#d62728', linewidth=2.5, label='Scenario A: early bear market (losses at the start)')
ax2.plot(path_b_ext, color='#1f77b4', linewidth=2.5, label='Scenario B: late bear market (losses at the end)')
ax2.set_title(f'B: with annual withdrawal of {annual_withdrawal:,} € (SORR effect / decumulation)'.replace(',', '.'), 
             fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Portfolio value (€)', fontsize=10)
ax2.set_xlabel('Years', fontsize=10)

# --- General formatting ---
for ax in [ax1, ax2]:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(euro_format))
    ax.legend(loc='upper left', frameon=True, fontsize=9)
    ax.set_xlim(-0.5, years + 0.5)

# Main title
#plt.suptitle('Figure 1: schematic illustration of SORR in the withdrawal phase', 
             #fontsize=15, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Save figure (optional)
plt.savefig('../assets/SORR_schema.png', dpi=300)

plt.show()